In [ ]:
%load_ext autoreload
%autoreload 2
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import clear_output

import sys
sys.path.append('/home/mattia/Desktop/Repos/posebench/benchmarks_3D')
from benchmark_pose import eval_colmap_model_all_scenes

models = [
    "vggt", 
    "vggt_ba", 
    "vggt_edge"
] 

## Helpers

In [ ]:
def read_results(dataset, target_folder, models, thrs=[5], fill_auc_nan_with_zeros=False):
    base_target = f"/home/mattia/Desktop/datasets/{dataset}"
    base_repo = "/home/mattia/Desktop/Repos/batchsfm/benchmark"
    scenes = os.listdir(base_target)

    # Read results
    dfs = {}
    for name in models:
        dfs[name] = eval_colmap_model_all_scenes(
            target_path=base_target,
            target_folder=target_folder,
            input_path=f"{base_repo}/{name}/{dataset}",
            input_folder="sparse",
            return_df=True,
            thrs=thrs,
            verbose=False
        )

    keys = sorted(list(dfs.keys()))
    for key in keys:
        dfs[key].columns = [f"{col}_{key}" if "auc" in col else col for col in dfs[key].columns]
    
    # Read timings
    total_timings = {}
    for model in models:
        if model not in total_timings:
                total_timings[f"{model}"] = {}
        for scene in scenes:
            recon_path = f"benchmark/{model}/{dataset}/{scene}"
            if not os.path.exists(recon_path):
                continue
            if scene not in total_timings[f"{model}"]:
                total_timings[f"{model}"][scene] = None
                try:
                    with open(f"{recon_path}/sparse/timings.txt", "r") as f:
                        lines = f.readlines()
                    total_timings[f"{model}"][scene] = float(lines[-1].split()[1])
                except FileNotFoundError:
                    
                    total_timings[f"{model}"][scene] = None

    df = pd.concat([dfs[key][f'auc@5_{key}'] for key in keys], axis=1)
    if fill_auc_nan_with_zeros:
        df.fillna(0, inplace=True)

    df = pd.concat([df, pd.DataFrame(total_timings)], axis=1)
    if "vggt_edge" in df.columns:
        df['vggt_edge'] += df['vggt']  # include base model time
    
    df.columns = [col.replace("_", "+") if "auc" not in col else col for col in df.columns]
    # df.dropna(inplace=True) 
    df.loc['mean'] = df.mean()
    df = df.round(2)

    # clean up cell prints
    clear_output()
    print("Results:")
    display(df)

    return df

In [ ]:
def plot_auc5_with_time(df, dataset, models):
    # --- 1. Data Preparation ---
    df_auc = df[[col for col in df.columns if 'auc@5' in col]].copy()
    
    # Clean column names
    clean_cols = [col.replace('auc@5_', '') for col in df_auc.columns]
    df_auc.columns = clean_cols
    
    # Create the Time row
    time_dict = {}
    for model in models:
        time_dict[model] = df[model].mean() 
    time_values = [time_dict.get(col, 0) for col in clean_cols]
    df_time_row = pd.DataFrame([time_values], columns=clean_cols, index=['Time'])
    
    # Combine into one DataFrame
    df_combined = pd.concat([df_auc, df_time_row])

    # --- 2. Masking for Dual Axis ---
    df_p1 = df_combined.copy()
    df_p1.loc['Time'] = np.nan # Hide Time on Left Axis
    
    df_p2 = df_combined.copy()
    df_p2.iloc[:-1] = np.nan   # Hide AUC on Right Axis

    # --- 3. Plotting ---
    fig, ax1 = plt.subplots(figsize=(len(df_combined)/1.1, 4))
    
    # Plot AUC (Left Axis)
    df_p1.plot(kind='bar', ax=ax1, width=0.8)
    
    # Plot Time (Right Axis)
    ax2 = ax1.twinx()
    df_p2.plot(kind='bar', ax=ax2, width=0.8, legend=False)

    # --- 4. Styling & Separator ---
    ax1.set_ylabel('AUC@5')
    ax2.set_ylabel('Time (s)')
    ax1.set_title(f'AUC@5 & Time - {dataset[:1].upper()+dataset[1:]} Dataset')

    # Black Dashed Line between AUC rows and Time row
    sep_x = len(df_auc) - 0.5
    ax1.axvline(x=sep_x, color='black', linestyle='--', linewidth=1.5, alpha=0.7)

    # --- ROTATION CHANGE HERE ---
    # Rotate 45 degrees and align right so the text ends at the tick mark
    ax1.set_xticklabels(df_combined.index, rotation=45, ha='right')

    # --- 5. Legend Placement ---
    ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)

    plt.tight_layout()
    plt.show()

In [ ]:
.

## ETH3D

In [ ]:
dataset = "eth3d"
df = read_results(dataset, "sparse", models, fill_auc_nan_with_zeros=False)

In [ ]:
plot_auc5_with_time(df, dataset, models)

In [ ]:
.

## 7Scenes

In [ ]:
dataset = "7scenes"
df = read_results(dataset, "colmap/sparse", models, fill_auc_nan_with_zeros=False)

In [ ]:
plot_auc5_with_time(df, dataset, models)

In [ ]:
.

## Mip-NeRF360

In [ ]:
dataset = "mipnerf360"
df = read_results(dataset, "sparse_150", models, fill_auc_nan_with_zeros=False)

In [ ]:
plot_auc5_with_time(df, dataset, models)

## TerraSky3D Test set

In [ ]:
dataset = "terrasky3D"
df = read_results(dataset, "sparse", models, fill_auc_nan_with_zeros=False)

In [ ]:
plot_auc5_with_time(df, dataset, models)

In [ ]:
.

## Tank and Temples

In [ ]:
dataset = "tt" # tanks_and_temples 
df = read_results(dataset, "sparse_150", models, fill_auc_nan_with_zeros=False)

In [ ]:
plot_auc5_with_time(df, dataset, models)